# 持续偏航运动分支

原世界x坐标在本分支改为累计前向行程 **l**，$\dot l=v$。旧物理参数l（车轴到质心距离）改记 **$l_c$**，数值代码为`com_height`。偏航角沿用psi，等同本次需求中的phi。

本笔记本独立于旧静止点笔记本，调用同目录的六坐标精确拉格朗日投影脚本。完整解释见README.md。

In [1]:
from pathlib import Path
import runpy
import sympy as sm
from IPython.display import display, Math
candidates=[Path.cwd()/"derive.py", Path.cwd()/"theory/yaw_motion/derive.py"]
source=next(p for p in candidates if p.exists())
model=runpy.run_path(str(source))
sm.init_printing()


PASS: exact Lagrange projection, nullspace and power identity.
M= Matrix([[2*J_s/r**2 + m_B + 2*m_W, 0, l_c*m_B*cos(theta)], [0, I_x*sin(theta)**2 - I_z*sin(theta)**2 + I_z + J_s*d**2/(2*r**2) + 2*J_t + d**2*m_W/2 + l_c**2*m_B*sin(theta)**2, 0], [l_c*m_B*cos(theta), 0, I_y + l_c**2*m_B]])
h= Matrix([[-l_c*m_B*(omega**2 + p_rate**2)*sin(theta)], [omega*(2*I_x*p_rate*cos(theta) - 2*I_z*p_rate*cos(theta) + 2*l_c**2*m_B*p_rate*cos(theta) + l_c*m_B*v)*sin(theta)], [(-I_x*omega**2*cos(theta) + I_z*omega**2*cos(theta) - g*l_c*m_B - l_c**2*m_B*omega**2*cos(theta))*sin(theta)]])
B= Matrix([[1/r, 1/r], [-d/(2*r), d/(2*r)], [-1, -1]])


## 六坐标允许速度映射 S

In [2]:
display(model["S"])

⎡cos(ψ)   0   0⎤
⎢              ⎥
⎢sin(ψ)   0   0⎥
⎢              ⎥
⎢  0      1   0⎥
⎢              ⎥
⎢  0      0   1⎥
⎢              ⎥
⎢  1     -d    ⎥
⎢  ─     ───  0⎥
⎢  r     2⋅r   ⎥
⎢              ⎥
⎢  1      d    ⎥
⎢  ─     ───  0⎥
⎣  r     2⋅r   ⎦

## 完整动能（约束代入前）

In [3]:
display(model["T"])

## 势能（略去常数）

In [4]:
display(model["V"])

## 新质量矩阵 M(theta)

In [5]:
display(model["expected_M"])

⎡2⋅Jₛ                                                                          ↪
⎢──── + m_B + 2⋅m_W                               0                            ↪
⎢  2                                                                           ↪
⎢ r                                                                            ↪
⎢                                                                              ↪
⎢                              2           2                                   ↪
⎢                          Jₛ⋅d           d ⋅m_W   ⎛              2    ⎞    2  ↪
⎢        0           I_z + ───── + 2⋅Jₜ + ────── + ⎝Iₓ - I_z + l_c ⋅m_B⎠⋅sin ( ↪
⎢                             2             2                                  ↪
⎢                          2⋅r                                                 ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎣  l_c⋅m_B⋅cos(θ)           

## 偏航耦合与重力项 h

In [6]:
display(model["expected_h"])

⎡                             ⎛ 2        2⎞                            ⎤
⎢                    -l_c⋅m_B⋅⎝ω  + pᵣₐₜₑ ⎠⋅sin(θ)                     ⎥
⎢                                                                      ⎥
⎢  ⎛                         ⎛                    2    ⎞              ⎞⎥
⎢ω⋅⎝l_c⋅m_B⋅v⋅sin(θ) + pᵣₐₜₑ⋅⎝2⋅Iₓ - 2⋅I_z + 2⋅l_c ⋅m_B⎠⋅sin(θ)⋅cos(θ)⎠⎥
⎢                                                                      ⎥
⎢                          2 ⎛               2    ⎞                    ⎥
⎣     -g⋅l_c⋅m_B⋅sin(θ) + ω ⋅⎝-Iₓ + I_z - l_c ⋅m_B⎠⋅sin(θ)⋅cos(θ)      ⎦

## 电机力矩映射 B_tau

In [7]:
display(model["B"])

⎡ 1    1 ⎤
⎢ ─    ─ ⎥
⎢ r    r ⎥
⎢        ⎥
⎢-d    d ⎥
⎢───  ───⎥
⎢2⋅r  2⋅r⎥
⎢        ⎥
⎣-1   -1 ⎦

## 零速度回归与数值验证

`simulation/yaw_motion/design_and_check.py`验证零速度A/B/K回归旧分支，并检验移动参考雅可比和三组精确非线性积分；`check_projection.py`验证30个随机状态下独立六坐标乘子系统与新方程一致。

Gazebo启动与物理结果见`ros2_ws/YAW_MOTION.md`。累计行程l不与世界x混用，车体质心高度不会被当成状态量。